# Notebook 03: Parameter-Efficient Fine-Tuning with LoRA

### Research Questions:
> **RQ2:** *How much does LoRA improve financial sentiment classification over the base model?*
> **RQ5:** *How many parameters actually need to be trained?*

### Mathematical Concept of LoRA

Standard fine-tuning updates the entire weight matrix: $W' = W + \Delta W$.

Instead of storing updates for all parameters, LoRA freezes $W$ and decomposes $\Delta W$ into two low-rank matrices $B$ and $A$:

$$\Delta W = \frac{\alpha}{r} (B \cdot A)$$

The effective forward weight is:

$$\boxed{W' = W + \frac{\alpha}{r} B A}$$

- $W \in \mathbb{R}^{d \times k}$ is frozen (requires no optimizer states or gradients).
- $A \in \mathbb{R}^{r \times k}$ is initialized randomly from a Gaussian distribution.
- $B \in \mathbb{R}^{d \times r}$ is initialized to zero, ensuring $\Delta W = 0$ at the start.
- $r=16$ is the low rank, and $\alpha=32$ is the scaling factor.
- Over **99%** of the parameters remain completely frozen!

In [ ]:
# === MOUNT GOOGLE DRIVE & SETUP ===
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment')
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'pyyaml'],
    check=True)

for d in ['results/metrics', 'results/predictions', 'results/figures',
          'results/logs', 'adapters/lora', 'adapters/qlora']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'src/ found: {os.path.exists("src")}')  # Must be True!
print('Setup complete! ✅')

Mounted at /content/drive
Working directory: /content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment
src/ found: True
Setup complete! ✅


In [ ]:
# Step 2: Hardware check and feasibility check
import json
import torch


from src.model_utils import detect_hardware, print_hardware_summary, check_vram_feasibility

hw = detect_hardware()
print_hardware_summary(hw)
# Primary research model: Qwen2.5-3B-Instruct
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
is_feasible, warning_msg = check_vram_feasibility(MODEL_NAME, method='lora', hardware_info=hw)
print(warning_msg)
print(f'Active model: {MODEL_NAME} for 16-bit LoRA fine-tuning.')

           HARDWARE & RUNTIME ENVIRONMENT SUMMARY           
CUDA Available:        True
GPU Name:              Tesla T4
GPU VRAM:              14.56 GB (14912.69 MB)
CUDA Version:          12.8
BF16 Supported:        True
PyTorch Version:       2.11.0+cu128
Transformers Version:  5.16.1
Compute Dtype:         bfloat16
Hardware check PASSED: 14.56 GB VRAM is sufficient for Qwen/Qwen2.5-3B-Instruct (LORA).
Active model: Qwen/Qwen2.5-3B-Instruct for 16-bit LoRA fine-tuning.


In [ ]:
# Step 3: Load dataset and prepare splits
from src.data_utils import load_raw_dataset, create_stratified_split, prepare_hf_training_dataset
from src.model_utils import load_tokenizer

tokenizer = load_tokenizer(MODEL_NAME)
df = load_raw_dataset('takala/financial_phrasebank', 'sentences_50agree')
train_df, val_df, test_df = create_stratified_split(df, 0.70, 0.15, 0.15, random_seed=42)

train_dataset = prepare_hf_training_dataset(train_df, tokenizer=tokenizer, max_length=256)
val_dataset = prepare_hf_training_dataset(val_df, tokenizer=tokenizer, max_length=256)
print(f'Train dataset prepared: {len(train_dataset)} examples.')
print(f'Validation dataset prepared: {len(val_dataset)} examples.')

Loading via `datasets` failed (Dataset scripts are no longer supported, but found financial_phrasebank.py). Using direct Hugging Face Hub download fallback...
Train dataset prepared: 3392 examples.
Validation dataset prepared: 727 examples.


In [ ]:
import sys

# 1. Fake the version string on the loaded torchao module in memory
if 'torchao' in sys.modules:
    sys.modules['torchao'].__version__ = '0.16.0'

# 2. Patch every single submodule in memory that imported is_torchao_available
for mod in list(sys.modules.values()):
    if hasattr(mod, 'is_torchao_available'):
        setattr(mod, 'is_torchao_available', lambda *a, **k: False)

print("TorchAO safely bypassed! ✅")

TorchAO safely bypassed! ✅


In [ ]:
# Step 4: Load Base Model and Apply LoRA

from src.model_utils import load_base_model, apply_lora_to_model, count_parameters
from src.memory_utils import MemoryTracker, get_gpu_memory_stats, reset_memory_stats

reset_memory_stats()
print(f'Loading base model {MODEL_NAME}...')
base_model = load_base_model(MODEL_NAME, quantization=False)

print('Applying LoRA adapter (r=16, alpha=32, target_modules=q,k,v,o)...')
model = apply_lora_to_model(
    base_model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    is_qlora=False,
    gradient_checkpointing=True,
)

param_stats = count_parameters(model)
print('=== Parameter Efficiency Analysis ===')
print(f'Total Parameters:     {param_stats["total_parameters_millions"]} M')
print(f'Trainable Parameters: {param_stats["trainable_parameters_millions"]} M')
print(f'Trainable Percentage: {param_stats["trainable_percentage"]} %')

Loading base model Qwen/Qwen2.5-3B-Instruct...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Applying LoRA adapter (r=16, alpha=32, target_modules=q,k,v,o)...
=== Parameter Efficiency Analysis ===
Total Parameters:     3093.31 M
Trainable Parameters: 7.3728 M
Trainable Percentage: 0.2383 %


In [ ]:
# Tokenize train and validation datasets
def tokenize_dataset(ds):
    return ds.map(
        lambda e: tokenizer(e["text"], truncation=True, max_length=256),
        batched=True,
        remove_columns=[col for col in ds.column_names if col not in ["input_ids", "attention_mask"]]
    )

train_dataset = tokenize_dataset(train_dataset)
val_dataset = tokenize_dataset(val_dataset)
print("Tokenized columns:", train_dataset.column_names)  # Should show: ['input_ids', 'attention_mask']

Map:   0%|          | 0/3392 [00:00<?, ? examples/s]

Map:   0%|          | 0/727 [00:00<?, ? examples/s]

Tokenized columns: ['input_ids', 'attention_mask']


### Colab runtime-resume support

Training checkpoints are stored on Google Drive under `checkpoints/lora`. If Colab disconnects or the runtime is restarted, rerun the notebook setup/model/trainer cells and the training cell will automatically detect the latest checkpoint and resume from it. The latest two checkpoints are retained for safety.


In [ ]:
# Step 5: Configure Training and Run Fine-Tuning
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import time

# checkpoint

CHECKPOINT_DIR = os.path.join(
    '/content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment',
    'checkpoints', 'lora'
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # Effective batch size = 8
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,  # latest 2 checkpoints
    bf16=hw['bf16_supported'],
    fp16=not hw['bf16_supported'] and hw['cuda_available'],
    gradient_checkpointing=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# for restart and checkpoint check ===

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint is not None:
    print(f'Latest checkpoint found: {last_checkpoint}')
    print('Resuming LoRA training from the latest checkpoint...')
else:
    print('No existing checkpoint found.')
    print('Starting LoRA training from scratch...')

print('Starting LoRA training with memory tracking...')
with MemoryTracker('LoRA Training') as train_tracker:
    t0 = time.perf_counter()
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    training_duration = time.perf_counter() - t0

print(f'Training completed in {training_duration:.1f} seconds ({training_duration/60:.2f} minutes).')
print(train_tracker.summary())

Latest checkpoint found: /content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment/checkpoints/lora/checkpoint-1272
Resuming LoRA training from the latest checkpoint...
Starting LoRA training with memory tracking...


Epoch,Training Loss,Validation Loss


Training completed in 5.4 seconds (0.09 minutes).
[LoRA Training] Peak Allocated: 11857.44 MB (11.58 GB) | Peak Reserved: 11928.0 MB (11.648 GB)


In [ ]:
# Step 6: Save LoRA Adapter and Check Size
from src.model_utils import get_adapter_disk_size_mb

adapter_dir = os.path.join('/content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment', 'adapters', 'lora')
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

adapter_size = get_adapter_disk_size_mb(adapter_dir)
print(f'LoRA adapter saved to {adapter_dir}')
print(f'Total adapter disk size: {adapter_size} MB (vs ~15,000 MB full model weights!)')

LoRA adapter saved to /content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment/adapters/lora
Total adapter disk size: 39.06 MB (vs ~15,000 MB full model weights!)


In [ ]:
# Step 7: Evaluate Fine-Tuned Model on Fixed Test Set
from src.data_utils import format_chat_prompt
from src.metrics import parse_model_output, compute_classification_metrics, format_metrics_summary
from src.visualization import plot_confusion_matrix_heatmap
import numpy as np
device = next(model.parameters()).device
raw_outputs = []
parsed_preds = []
latencies = []

model.eval()
print(f'Evaluating fine-tuned LoRA model on {len(test_df)} test samples...')

for idx, row in test_df.iterrows():
    prompt = format_chat_prompt(row['sentence'], tokenizer=tokenizer)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000.0)

    gen_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    raw_outputs.append(gen_text)
    parsed_preds.append(parse_model_output(gen_text))

metrics = compute_classification_metrics(test_df['label_text'].tolist(), parsed_preds)
metrics['method'] = 'lora'
metrics['model_name'] = MODEL_NAME
metrics['trainable_parameters'] = param_stats['trainable_parameters']
metrics['trainable_percentage'] = param_stats['trainable_percentage']
metrics['adapter_size_mb'] = adapter_size
metrics['training_time_seconds'] = round(training_duration, 2)
metrics['peak_vram_gb'] = train_tracker.peak_allocated_gb
metrics['latency_median_ms'] = round(float(np.median(latencies)), 2)

print(format_metrics_summary(metrics, 'LoRA Fine-Tuned Model'))

with open('results/metrics/lora.json', 'w') as f:
    json.dump(metrics, f, indent=2)

test_df_copy = test_df.copy()
test_df_copy['predicted_label'] = parsed_preds
test_df_copy['raw_output'] = raw_outputs
test_df_copy.to_csv('results/predictions/lora.csv', index=False)

plot_confusion_matrix_heatmap(metrics['confusion_matrix'], method_name='LoRA Fine-Tuned')
print('Saved results to results/metrics/lora.json and results/predictions/lora.csv')

Evaluating fine-tuned LoRA model on 727 test samples...
        CLASSIFICATION PERFORMANCE: LORA FINE-TUNED MODEL
Accuracy:            0.8583 (85.83%)
Macro F1:            0.8538
Macro Precision:     0.8412
Macro Recall:        0.8702
Weighted F1:         0.8578
Invalid Output Rate: 0.00% (0/727)
-------------------------------------------------------
Per-Class Performance:
  Negative  ->  P: 0.8208 | R: 0.9560 | F1: 0.8832 (n=91)
  Neutral   ->  P: 0.8873 | R: 0.8750 | F1: 0.8811 (n=432)
  Positive  ->  P: 0.8154 | R: 0.7794 | F1: 0.7970 (n=204)
Saved results to results/metrics/lora.json and results/predictions/lora.csv


### Key Takeaways from LoRA Fine-Tuning

- **Parameter Efficiency:** We only trained approximately ~0.1% to 0.3% of the total parameter count.
- **Checkpoint Size:** The adapter checkpoint is only a few megabytes compared to the full multi-gigabyte base model.
- In the next notebook, we introduce **QLoRA (4-bit quantization)** to dramatically reduce the VRAM footprint while keeping the exact same LoRA adapter setup.